<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/05-instruction-set-and-assembly.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Instruction Set Architecture and Assembly** {#instruction-set-architecture-and-assembly}

The previous chapters built the physical ideas needed for computation: numbers, combinational functions, stored state, and timing. Software does not normally control those gates and flip-flops directly. It uses an **instruction set architecture** (ISA), a stable contract that describes which machine operations exist and how they change programmer-visible state.

This chapter uses **RV32I**, the 32-bit RISC-V base integer ISA, as its running example. RV32I is small enough to encode and simulate by hand, yet complete enough to be a compiler target. The focus is not memorizing every opcode. The goal is to read an instruction as a precise state transformation, understand how assembly represents that transformation symbolically, and distinguish ISA rules from ABI and microarchitectural choices.

### **What Is an Instruction Set Architecture?** {#what-is-an-instruction-set-architecture}

An ISA defines the behavior a processor promises to software. It normally specifies:

- programmer-visible registers and their widths;
- instruction encodings and operation semantics;
- how memory is addressed and which data widths can be transferred;
- control-flow, exception, and privilege behavior;
- atomicity and ordering rules needed for communication;
- optional extensions and the rules by which they compose.

![The ISA fixes observable behavior while allowing very different processor organizations to implement the same contract.](assets/isa-contract-boundary.svg){fig-align="center" width="100%"}

If two processors correctly implement the same ISA and execution environment, the same machine-code program should observe the same architectural results even if one processor is a small multi-cycle core and another is a deeply pipelined out-of-order design. Their clock rates, cache sizes, internal registers, speculation policies, and execution-unit counts are **organization**, not ISA.

It is helpful to separate four nearby concepts:

| Layer | What it defines | RISC-V example |
|---|---|---|
| ISA | encoded operations and architectural state transitions | `add x5, x6, x7` adds two registers |
| assembly language | human-readable notation accepted by an assembler | mnemonic `add`, labels, directives |
| ABI | binary conventions that let separately compiled code cooperate | `a0-a7` arguments, `sp` alignment, saved registers |
| microarchitecture | hardware organization that realizes the ISA | single-cycle, pipelined, or out-of-order core |

Assembly is not the ISA itself. One machine instruction can have several accepted textual aliases, and one pseudoinstruction can expand into several machine instructions. Likewise, the ISA calls integer registers `x0-x31`; names such as `ra`, `sp`, and `a0` are assigned by the software calling convention.

An instruction can be described as an architectural transition

$$
A_{k+1}=I(A_k,M_k),
$$

where $A_k$ is visible architectural state before instruction $k$, $M_k$ is the relevant memory state, $I$ is the decoded instruction semantics, and $A_{k+1}$ is the committed visible state afterward. A speculative processor may execute many internal operations and later discard some of them, but it must retire instructions as though this architectural contract were followed.

The ISA therefore creates compatibility and abstraction at the same time. Software gains a stable target, while hardware designers remain free to improve implementation below the boundary.

### **Programmer-Visible Machine State** {#programmer-visible-machine-state}

Programmer-visible state is the information an instruction may read or modify according to the ISA. RV32I exposes 32 integer registers and a program counter to unprivileged code. Instructions can also access a byte-addressed memory space through loads and stores. Privileged control and status registers exist in broader RISC-V systems, but they are outside the base RV32I state considered here.

![RV32I exposes thirty-two 32-bit integer registers, a program counter, and byte-addressed memory; ABI aliases are a separate software layer.](assets/rv32i-programmer-state.svg){fig-align="center" width="100%"}

#### **Registers** {#registers}

In RV32I, `XLEN=32`: each integer register contains 32 bits. Register `x0` is special because every read returns zero and every write is discarded. Registers `x1-x31` are ordinary bit containers. An instruction determines whether a pattern is interpreted as signed two's-complement, unsigned, an address, or Boolean fields.

The usual three-register arithmetic form is

```text
operation rd, rs1, rs2
```

`rs1` and `rs2` name source registers, while `rd` names the destination. For example,

```text
add x5, x6, x7
```

means

$$
x[5]\leftarrow(x[6]+x[7])\bmod 2^{32}.
$$

The brackets mean "contents of register." Modulo $2^{32}$ keeps the low 32 result bits and discards carry beyond XLEN. Unlike some ISAs, base RISC-V integer arithmetic does not update a condition-code register. Comparisons and branches name their operands explicitly.

The immutable zero register is useful because it turns ordinary instructions into common operations: `addi rd, x0, imm` creates a small constant, `add rd, rs, x0` copies a register, and writing `rd=x0` discards a result without adding separate encodings.

<details>
<summary>Python model: RV32 register state and the hardwired zero register</summary>

~~~python
class RV32Registers:
    def __init__(self) -> None:
        self._x = [0] * 32
        self.mask = (1 << 32) - 1

    def read(self, index: int) -> int:
        if not 0 <= index < 32:
            raise IndexError("register index must be in x0-x31")
        return 0 if index == 0 else self._x[index]

    def write(self, index: int, value: int) -> None:
        if not 0 <= index < 32:
            raise IndexError("register index must be in x0-x31")
        if index != 0:
            self._x[index] = value & self.mask


registers = RV32Registers()
registers.write(6, 0xFFFF_FFFF)
registers.write(7, 2)
registers.write(5, registers.read(6) + registers.read(7))
assert registers.read(5) == 1       # Arithmetic wraps modulo 2^32.

registers.write(0, 12345)
assert registers.read(0) == 0       # x0 never changes.
~~~

</details>

#### **Program Counter** {#program-counter}

The **program counter** (`pc`) contains the address of the current instruction. It determines where instruction fetch occurs, but ordinary RV32I instructions do not name `pc` as one of `x0-x31`.

For a normally sequenced 32-bit instruction,

$$
pc_{next}=pc+4.
$$

Four is the instruction length in bytes, not bits. The optional compressed extension introduces 16-bit instructions, so a core supporting it may advance by 2 for a compressed instruction and by 4 for an ordinary instruction.

Control-flow instructions replace sequential progression:

- a conditional branch chooses between `pc+4` and a PC-relative target;
- `JAL` jumps to a PC-relative target and can save a return address;
- `JALR` forms a target from a register plus an immediate;
- an exception or interrupt transfers control according to privileged architectural rules.

The phrase "program counter" can hide implementation detail. A pipeline may hold several internal PCs for instructions in different stages, and an out-of-order core may predict future PCs. The architectural effect must still match the ISA when instructions commit.

PC-relative addressing is valuable because code can move in memory without rewriting every internal absolute address. Labels in assembly eventually become offsets, often accompanied by relocation records when the final address is not yet known.

#### **Memory and Address Space** {#memory-and-address-space}

RV32I has a 32-bit byte-addressed address space. An address identifies one byte, so the architectural range contains up to $2^{32}$ byte addresses. This does not promise that a machine installs 4 GiB of physical RAM: regions may be unmapped, protected, reserved, or connected to memory-mapped devices.

Multi-byte values occupy consecutive addresses. In a little-endian execution environment, the least-significant byte of a word is stored at the lowest address. For the word `0x12345678` beginning at address `A`:

| Address | Stored byte |
|---|---|
| `A` | `0x78` |
| `A+1` | `0x56` |
| `A+2` | `0x34` |
| `A+3` | `0x12` |

Endianness changes byte order in memory, not the bit numbering inside a register. A byte load from address `A` still returns the byte placed at `A`.

An access is **naturally aligned** when its address is a multiple of its size: a 4-byte word is naturally aligned at addresses divisible by 4. The execution environment determines how misaligned loads and stores are handled. They may complete, trap visibly, or be supported through an invisible mechanism. Portable and performant code normally keeps scalar objects naturally aligned.

Address space is also an interface boundary. A normal RAM location stores data, while a memory-mapped device address may trigger input/output behavior. The same load/store instruction syntax can access both, but caching, ordering, permissions, and side effects depend on the platform and privileged architecture.

### **RISC-V Instruction Formats and Encoding** {#risc-v-instruction-formats-and-encoding}

An instruction word is not just a numeric identifier. Its bit fields tell the decoder which operation to perform and where its operands are located. Base RV32I uses fixed 32-bit instructions and four core layouts: R, I, S, and U. Branches and jumps use B and J immediate variants built from the same field positions.

::: {layout-ncol=1}
![R-type: two register sources and one destination.](assets/riscv-format-1.svg)

![I-type: one register source, one destination, and a 12-bit immediate.](assets/riscv-format-2.svg)

![S-type: two register sources and a split 12-bit store immediate.](assets/riscv-format-3.svg)

![U-type: a destination and a 20-bit upper immediate.](assets/riscv-format-4.svg)
:::

*Image source: [RISC-V Unprivileged ISA Specification, RV32I Base Instruction Formats](https://docs.riscv.org/reference/isa/v20260120/unpriv/rv32.html), RISC-V International, CC BY 4.0.*

The fields have stable roles:

| Field | Width | Purpose |
|---|---:|---|
| `opcode` | 7 bits | selects a broad instruction class and format |
| `rd` | 5 bits | names one of 32 destination registers |
| `funct3` | 3 bits | distinguishes operations inside an opcode class |
| `rs1` | 5 bits | names first source register |
| `rs2` | 5 bits | names second source register |
| `funct7` | 7 bits | provides additional operation selection for R-type forms |
| immediate | format-dependent | embeds a constant, address offset, or control-flow displacement |

RISC-V keeps `rd`, `rs1`, and `rs2` in the same bit positions whenever those fields exist. That regularity simplifies and accelerates register decoding. The tradeoff is that S-, B-, and J-type immediates are split or rearranged in the instruction word.

The formats are selected by operand needs:

- **R-type:** register-register arithmetic such as `add` and `sub`;
- **I-type:** register-immediate arithmetic, loads, and `JALR`;
- **S-type:** stores, which need two sources but no destination register;
- **B-type:** conditional branches with a signed PC-relative offset;
- **U-type:** `LUI` and `AUIPC`, which provide upper immediate bits;
- **J-type:** `JAL`, with a wider signed PC-relative offset.

Most immediate values are sign-extended. For an $n$-bit encoded value $u$,

$$
sext_n(u)=
\begin{cases}
u,&u<2^{n-1},\\
u-2^n,&u\ge2^{n-1}.
\end{cases}
$$

$2^{n-1}$ is the sign-bit weight. Values below it are nonnegative; values at or above it have sign bit 1 and represent a negative two's-complement number after subtracting $2^n$. RISC-V places the sign source for regular immediates at instruction bit 31 so sign extension can begin in parallel with decoding.

For `add x5, x6, x7`, the fields are:

```text
funct7  rs2    rs1    funct3  rd     opcode
0000000 00111  00110  000     00101  0110011
```

Concatenating them gives `0x007302B3`. The hexadecimal value is not separately assigned to the instruction; it is simply the packed result of all fields.

<details>
<summary>Python encoder: construct and inspect RV32I R-type and I-type words</summary>

~~~python
def check_register(index: int) -> None:
    if not 0 <= index < 32:
        raise ValueError("register index must fit five bits")


def encode_r(funct7: int, rs2: int, rs1: int,
             funct3: int, rd: int, opcode: int) -> int:
    for register in (rd, rs1, rs2):
        check_register(register)
    if not 0 <= funct7 < 128 or not 0 <= funct3 < 8 or not 0 <= opcode < 128:
        raise ValueError("function or opcode field does not fit")
    return (
        (funct7 << 25)
        | (rs2 << 20)
        | (rs1 << 15)
        | (funct3 << 12)
        | (rd << 7)
        | opcode
    )


def encode_i(immediate: int, rs1: int, funct3: int,
             rd: int, opcode: int) -> int:
    check_register(rd)
    check_register(rs1)
    if not -2048 <= immediate <= 2047:
        raise ValueError("I-type immediate must fit signed 12 bits")
    imm12 = immediate & 0xFFF
    return (imm12 << 20) | (rs1 << 15) | (funct3 << 12) | (rd << 7) | opcode


add_x5_x6_x7 = encode_r(0b0000000, 7, 6, 0b000, 5, 0b0110011)
addi_x5_x6_minus12 = encode_i(-12, 6, 0b000, 5, 0b0010011)

assert add_x5_x6_x7 == 0x007302B3
assert addi_x5_x6_minus12 == 0xFF430293

# Common fields can be recovered from fixed positions.
assert (add_x5_x6_x7 >> 7) & 0x1F == 5   # rd
assert (add_x5_x6_x7 >> 15) & 0x1F == 6  # rs1
assert (add_x5_x6_x7 >> 20) & 0x1F == 7  # rs2
~~~

</details>

### **Arithmetic and Logical Instructions** {#arithmetic-and-logical-instructions}

RV32I arithmetic instructions transform register values and write one register result. Register-register operations use R-type encoding; constant-operand versions use I-type encoding. Both normally name their destination explicitly.

| Family | Examples | Architectural effect |
|---|---|---|
| addition/subtraction | `add`, `sub`, `addi` | low XLEN bits of arithmetic result |
| comparison | `slt`, `sltu`, `slti`, `sltiu` | write 1 when comparison is true, otherwise 0 |
| Boolean logic | `and`, `or`, `xor` and immediate forms | bitwise operation |
| left shift | `sll`, `slli` | shift left and insert zeroes |
| logical right shift | `srl`, `srli` | shift right and insert zeroes |
| arithmetic right shift | `sra`, `srai` | shift right and replicate sign bit |

For register arithmetic,

$$
x[rd]\leftarrow f(x[rs1],x[rs2])\bmod2^{XLEN}.
$$

$f$ identifies the selected ALU operation. `XLEN` is 32 for RV32I and 64 for RV64I. The modulo matters for addition, subtraction, and left shift because high bits can leave the register width.

Signed and unsigned comparison use identical stored bit patterns but different interpretation. If `x5=0xFFFFFFFF`, then signed comparison interprets it as $-1$, while unsigned comparison interprets it as $2^{32}-1$. Thus `slt` and `sltu` can return different answers without changing either operand.

Shift instructions use only the low $\log_2(XLEN)$ bits of a register shift amount. RV32I therefore uses five bits because $\log_2(32)=5$. Logical and arithmetic right shift differ only in the inserted high bits; the latter preserves the two's-complement sign pattern.

Base RV32I does not include integer multiply or divide. Those operations belong to the standard `M` extension. An implementation lacking `M` can call software routines, while a processor supporting `M` exposes additional instructions under the same modular ISA framework.

RISC-V does not generate a special signed-overflow exception for base integer addition. It writes the wrapped result. Software that needs overflow detection performs explicit comparisons. This keeps arithmetic semantics uniform for signed and unsigned patterns.

<details>
<summary>Python model: execute representative RV32I ALU operations</summary>

~~~python
MASK32 = (1 << 32) - 1


def signed32(value: int) -> int:
    value &= MASK32
    return value - (1 << 32) if value & (1 << 31) else value


def rv32_alu(operation: str, left: int, right: int) -> int:
    left &= MASK32
    right &= MASK32
    shift = right & 0x1F  # RV32 uses the low five shift-amount bits.

    if operation == "add":
        result = left + right
    elif operation == "sub":
        result = left - right
    elif operation == "and":
        result = left & right
    elif operation == "or":
        result = left | right
    elif operation == "xor":
        result = left ^ right
    elif operation == "sll":
        result = left << shift
    elif operation == "srl":
        result = left >> shift
    elif operation == "sra":
        result = signed32(left) >> shift
    elif operation == "slt":
        result = int(signed32(left) < signed32(right))
    elif operation == "sltu":
        result = int(left < right)
    else:
        raise ValueError(f"unsupported operation: {operation}")

    return result & MASK32


assert rv32_alu("add", 0xFFFF_FFFF, 2) == 1
assert rv32_alu("slt", 0xFFFF_FFFF, 1) == 1    # signed -1 < 1
assert rv32_alu("sltu", 0xFFFF_FFFF, 1) == 0  # unsigned max > 1
assert rv32_alu("sra", 0x8000_0000, 4) == 0xF800_0000
assert rv32_alu("srl", 0x8000_0000, 4) == 0x0800_0000
~~~

</details>

### **Load, Store, and Addressing** {#load-store-and-addressing}

RISC-V is a **load-store architecture**. Arithmetic and logical instructions operate on registers; only loads and stores transfer values between registers and memory. Keeping memory access separate gives the datapath a regular boundary and makes address generation explicit.

Both loads and stores use base-plus-offset addressing:

$$
EA=(x[rs1]+sext_{12}(imm))\bmod2^{XLEN}.
$$

- $EA$ is the effective byte address.
- $x[rs1]$ is the base address held in a source register.
- $imm$ is the instruction's signed 12-bit offset.
- $sext_{12}$ extends that offset to XLEN bits.
- modulo $2^{XLEN}$ keeps address arithmetic within the architectural width.

![A base register and signed immediate form the effective address; loads move memory data to rd, while stores move rs2 data to memory.](assets/load-store-effective-address.svg){fig-align="center" width="100%"}

Assembly writes this mode as `offset(base)`:

```text
lw t0, 12(sp)    # t0 <- 32-bit word at address sp + 12
sw t1, 8(sp)     # memory word at sp + 8 <- low 32 bits of t1
```

A load has a destination register because data enters the register file. A store has no `rd`; its second source register supplies data. This is why store immediates occupy the bit positions that an R-type instruction would otherwise use partly for `rd`.

RV32I provides byte, halfword, and word access:

| Instruction | Bytes | Load extension or stored data |
|---|---:|---|
| `lb` | 1 | sign-extend byte to XLEN |
| `lbu` | 1 | zero-extend byte |
| `lh` | 2 | sign-extend halfword |
| `lhu` | 2 | zero-extend halfword |
| `lw` | 4 | load one 32-bit RV32 word |
| `sb` | 1 | store low 8 register bits |
| `sh` | 2 | store low 16 register bits |
| `sw` | 4 | store low 32 register bits |

Sign extension matters when loading a small signed integer. Byte `0xFF` becomes `0xFFFFFFFF` under `lb` but `0x000000FF` under `lbu`. The memory bytes are identical; only the register interpretation differs.

Arrays use a base address plus a scaled index. Because RV32I load/store instructions do not contain a general scale field, software computes the byte displacement explicitly. For a 4-byte integer array, index $i$ corresponds to `base + 4*i`, often produced with `slli index, index, 2` followed by addition.

<details>
<summary>Python memory model: little-endian loads, stores, and signed extension</summary>

~~~python
class LittleEndianMemory:
    def __init__(self, byte_count: int) -> None:
        self.data = bytearray(byte_count)

    def _check(self, address: int, width: int) -> None:
        if address < 0 or address + width > len(self.data):
            raise IndexError("memory access out of range")

    def store(self, address: int, value: int, width: int) -> None:
        self._check(address, width)
        for byte_index in range(width):
            self.data[address + byte_index] = (value >> (8 * byte_index)) & 0xFF

    def load(self, address: int, width: int, *, signed: bool = False) -> int:
        self._check(address, width)
        value = sum(
            self.data[address + byte_index] << (8 * byte_index)
            for byte_index in range(width)
        )

        # Sign-extend the selected access width into a 32-bit register.
        if signed and value & (1 << (8 * width - 1)):
            value -= 1 << (8 * width)
        return value & 0xFFFF_FFFF


memory = LittleEndianMemory(64)
memory.store(16, 0x12345678, width=4)
assert list(memory.data[16:20]) == [0x78, 0x56, 0x34, 0x12]
assert memory.load(16, 4) == 0x12345678

memory.store(24, 0xFF, width=1)
assert memory.load(24, 1, signed=True) == 0xFFFF_FFFF  # lb
assert memory.load(24, 1, signed=False) == 0x0000_00FF # lbu
~~~

</details>

### **Branches, Jumps, and Control Flow** {#branches-jumps-and-control-flow}

Control-flow instructions decide which instruction address becomes the next `pc`. Conditional branches implement decisions and loops; jumps implement unconditional transfers, procedure calls, and returns.

For a conditional branch,

$$
pc_{next}=
\begin{cases}
pc+sext(offset),&condition\text{ is true},\\
pc+4,&condition\text{ is false}.
\end{cases}
$$

The branch compares two registers directly. `beq` and `bne` test equality; `blt` and `bge` use signed order; `bltu` and `bgeu` use unsigned order. There is no hidden flag register from an earlier arithmetic instruction.

Branch offsets are PC-relative and encoded in multiples of two bytes. The low target bit is therefore implicit. This choice supports coexistence with 16-bit compressed instructions even though a base-only RV32I instruction is four bytes long.

`JAL` performs two architectural actions:

$$
x[rd]\leftarrow pc+4,
$$

$$
pc_{next}\leftarrow pc+sext(jump\ offset).
$$

Saving `pc+4` creates a return link. The conventional `jal ra, function` writes it into `x1` (`ra`), while `jal x0, label` discards the link and acts as a plain jump.

`JALR` uses a register-relative target:

$$
t=(x[rs1]+sext_{12}(imm)),
$$

$$
pc_{next}=t\mathbin{\&}\sim1.
$$

Clearing target bit 0 guarantees an even instruction address and leaves the low pointer bit available for auxiliary information in some software conventions. `jalr x0, 0(ra)` returns by jumping to the saved link and discarding a new one.

![Sequential execution, branches, JAL, and JALR provide alternative next-PC candidates; jumps can also write a return link.](assets/branch-jump-control-flow.svg){fig-align="center" width="100%"}

Branches create a hardware challenge addressed in Chapter 07: a pipeline may fetch several instructions before the condition and target are known. Prediction changes performance, but not the architectural branch semantics above.

<details>
<summary>Python model: calculate branch, JAL, and JALR effects</summary>

~~~python
MASK32 = (1 << 32) - 1


def branch(pc: int, left: int, right: int, offset: int, condition: str) -> int:
    comparisons = {
        "eq": left == right,
        "ne": left != right,
        "ltu": (left & MASK32) < (right & MASK32),
    }
    if condition not in comparisons:
        raise ValueError("unsupported branch condition")
    return (pc + (offset if comparisons[condition] else 4)) & MASK32


def jal(pc: int, offset: int) -> tuple[int, int]:
    link = (pc + 4) & MASK32
    target = (pc + offset) & MASK32
    return target, link


def jalr(pc: int, base: int, immediate: int) -> tuple[int, int]:
    link = (pc + 4) & MASK32
    target = (base + immediate) & MASK32
    return target & ~1, link


assert branch(0x1000, 5, 5, 24, "eq") == 0x1018
assert branch(0x1000, 5, 7, 24, "eq") == 0x1004
assert jal(0x2000, -16) == (0x1FF0, 0x2004)
assert jalr(0x3000, 0x4123, 4) == (0x4126, 0x3004)
~~~

</details>

### **Assembly Language** {#assembly-language}

Assembly language is a symbolic interface to machine instructions and toolchain features. It replaces binary fields with mnemonics and register names, uses labels instead of manually calculated addresses, and uses directives to describe sections and data. It remains architecture-specific and close to ISA behavior, but it is not a one-to-one transcription in every case.

A typical statement contains an optional label, an instruction or directive, operands, and an optional comment:

```text
loop:   addi t0, t0, -1    # decrement the loop counter
        bne  t0, zero, loop
```

The assembler determines register numbers, chooses encodings, expands pseudoinstructions, and records unresolved symbol references. Spacing and comment syntax are assembly-dialect conventions; operation semantics come from the ISA.

#### **Labels and Directives** {#labels-and-directives}

A **label** associates a symbolic name with the current location. It does not execute and does not necessarily allocate storage. The same label can later become a branch target, data address, or relocation expression.

An assembler **directive** begins with a dot and tells the assembler how to build the object file. It is not fetched by the processor as an instruction.

| Directive | Typical purpose |
|---|---|
| `.text` | select executable code section |
| `.data` | select initialized writable data section |
| `.bss` | select zero-initialized storage section |
| `.globl name` | make a symbol visible to the linker |
| `.align n` | align following content according to assembler rules |
| `.byte`, `.half`, `.word` | emit fixed-width data values |
| `.asciz` | emit a zero-terminated string |

For example:

```text
    .section .rodata
message:
    .asciz "hello"

    .text
    .globl read_first
read_first:
    la   t0, message
    lbu  a0, 0(t0)
    ret
```

`message` names data, while `read_first` names code. The `la` and `ret` lines are pseudoinstructions whose final machine sequence depends on context. The string bytes are object-file data, not instruction words.

Alignment directives deserve special care because syntax differs among assemblers and targets. GNU RISC-V `.align n` uses a power-of-two byte boundary, but portable source should follow the selected toolchain documentation rather than assume all assemblers use identical interpretation.

#### **Pseudoinstructions** {#pseudoinstructions}

A **pseudoinstruction** is assembler notation that expands into one or more real instructions. It improves readability without enlarging the hardware ISA.

| Pseudoinstruction | Typical base expansion | Meaning |
|---|---|---|
| `nop` | `addi x0, x0, 0` | no architectural data change |
| `mv rd, rs` | `addi rd, rs, 0` | copy register |
| `not rd, rs` | `xori rd, rs, -1` | invert every bit |
| `neg rd, rs` | `sub rd, x0, rs` | two's-complement negation |
| `j label` | `jal x0, label` | jump without link |
| `ret` | `jalr x0, 0(ra)` | return to link in `ra` |
| `li rd, constant` | one or more instructions | construct a constant |
| `la rd, symbol` | PC-relative instruction sequence | form a symbol address |

The phrase "typical expansion" matters. The assembler and linker can select different legal sequences according to code model, symbol location, extensions, position independence, and relaxation.

Loading a constant illustrates why expansion is not always trivial. A signed 12-bit value fits directly in `addi rd, x0, imm`. A wider 32-bit constant can use `lui` for upper bits and `addi` for signed low bits. Because `addi` sign-extends its 12-bit immediate, the split is commonly computed as

$$
hi20=(C+0x800)\gg12,
$$

$$
lo12=C-(hi20\ll12).
$$

- $C$ is the desired 32-bit constant.
- adding `0x800` rounds the high part upward when low bit 11 is 1;
- `\gg12` selects the adjusted upper 20 bits;
- $lo12$ becomes a signed value in the range $-2048$ through $2047$;
- `lui` supplies `hi20 << 12`, and `addi` supplies the signed remainder.

<details>
<summary>Python expansion helper: split a 32-bit constant for LUI plus ADDI</summary>

~~~python
def split_lui_addi(constant: int) -> tuple[int, int]:
    """Return a 20-bit LUI field and signed 12-bit ADDI immediate."""
    constant &= 0xFFFF_FFFF
    signed_constant = constant - (1 << 32) if constant & (1 << 31) else constant

    hi20 = (signed_constant + 0x800) >> 12
    lo12 = signed_constant - (hi20 << 12)
    if not -2048 <= lo12 <= 2047:
        raise AssertionError("adjusted low immediate must fit signed 12 bits")
    return hi20 & 0xFFFFF, lo12


def reconstruct(hi20: int, lo12: int) -> int:
    return ((hi20 << 12) + lo12) & 0xFFFF_FFFF


for constant in (0x0000_07FF, 0x0000_0800, 0x1234_5ABC, 0xFFFF_FFFF):
    high, low = split_lui_addi(constant)
    assert reconstruct(high, low) == constant

assert split_lui_addi(0x1234_5ABC) == (0x12346, -0x544)
~~~

</details>

When inspecting disassembly, decide whether the tool is showing canonical machine instructions or reconstructed pseudoinstructions. Two displays can describe the same bytes with different mnemonics.

#### **From Assembly to Machine Code** {#from-assembly-to-machine-code}

Assembly source becomes executable code through several stages. The assembler can immediately encode numeric operands and local relationships that are known, but final addresses often depend on other files and section layout.

![The assembler emits relocatable machine code and metadata; the linker resolves symbols and lays out an executable before the loader maps it for execution.](assets/assembly-toolchain.svg){fig-align="center" width="100%"}

The main artifacts are:

1. **Source file:** mnemonics, labels, directives, and expressions.
2. **Relocatable object:** encoded bytes, sections, symbols, and relocation records.
3. **Linked executable or library:** symbols resolved where possible and sections assigned final relationships.
4. **Loaded image:** segments mapped into an address space, permissions established, and entry state prepared.

A **symbol table** records names and their current section-relative values. A **relocation** says that particular bits must be adjusted when a symbol's final address becomes known. PC-relative RISC-V address sequences may use paired high and low relocations because no one instruction holds an arbitrary 32-bit address.

The linker can also perform **relaxation**: replace a general sequence with a shorter or simpler equivalent after final distances are known. This is why source-line count, object instruction count, and linked instruction count are not always identical.

A two-pass assembler illustrates label resolution. Pass one determines each instruction address and records labels. Pass two substitutes label-derived offsets and encodes final instruction words. Real assemblers additionally handle sections, macros, expressions, symbol visibility, relocations, and many instruction forms.

<details>
<summary>Python miniature: resolve labels and encode a small ADDI/BEQ program</summary>

~~~python
def encode_addi(rd: int, rs1: int, immediate: int) -> int:
    if not -2048 <= immediate <= 2047:
        raise ValueError("ADDI immediate out of range")
    return ((immediate & 0xFFF) << 20) | (rs1 << 15) | (rd << 7) | 0x13


def encode_beq(rs1: int, rs2: int, offset: int) -> int:
    if offset % 2 or not -4096 <= offset <= 4094:
        raise ValueError("BEQ offset must be an even signed 13-bit value")
    imm = offset & 0x1FFF
    return (
        (((imm >> 12) & 1) << 31)
        | (((imm >> 5) & 0x3F) << 25)
        | (rs2 << 20)
        | (rs1 << 15)
        | (((imm >> 1) & 0xF) << 8)
        | (((imm >> 11) & 1) << 7)
        | 0x63
    )


program = [
    ("start", "addi", (5, 0, 3)),
    (None, "addi", (5, 5, -1)),
    (None, "beq", (5, 0, "done")),
    (None, "beq", (0, 0, "start")),
    ("done", "addi", (10, 5, 0)),
]

# Pass 1: every base instruction is four bytes, so labels get addresses.
labels = {}
pc = 0
for label, _, _ in program:
    if label is not None:
        labels[label] = pc
    pc += 4

# Pass 2: encode, replacing symbolic branch targets with PC-relative offsets.
machine_words = []
pc = 0
for _, mnemonic, operands in program:
    if mnemonic == "addi":
        machine_words.append(encode_addi(*operands))
    elif mnemonic == "beq":
        rs1, rs2, target = operands
        machine_words.append(encode_beq(rs1, rs2, labels[target] - pc))
    pc += 4

assert labels == {"start": 0, "done": 16}
assert machine_words[2] == 0x00028463  # beq x5, x0, +8
assert machine_words[3] == 0xFE000AE3  # beq x0, x0, -12
~~~

</details>

### **Procedures and Calling Conventions** {#procedures-and-calling-conventions}

A procedure call must transfer control and later return, but independently compiled functions also need agreement about arguments, results, register ownership, stack layout, and alignment. RISC-V separates these concerns cleanly:

- `JAL` and `JALR` provide ISA-level control transfer and link behavior;
- the **procedure calling convention** in the processor-specific ABI assigns software roles to registers and the stack.

The ISA would allow a private program to use `x20` as a link register and grow a stack upward. Such code would still use valid instructions, but it could not safely call standard libraries unless it obeyed the same ABI at the boundary.

A conventional call uses

```text
jal ra, function
```

which saves `pc+4` in `ra` (`x1`). A return uses the `ret` pseudoinstruction, normally

```text
jalr x0, 0(ra)
```

A **leaf procedure** calls no other procedure and may leave `ra` in its incoming register. A **non-leaf procedure** will overwrite `ra` when it makes another call, so it normally saves the caller's return address first.

#### **Stack Frames** {#stack-frames}

The stack is a memory region managed through `sp` (`x2`). Under the standard RISC-V ABI it grows toward lower addresses, and `sp` is aligned to a 128-bit, or 16-byte, boundary on procedure entry. A procedure subtracts from `sp` to allocate a frame and adds the same size back before return.

![A possible sixteen-byte RV32 frame stores return and saved-register values above local slots; exact layout is procedure-specific.](assets/riscv-stack-frame.svg){fig-align="center" width="88%"}

*Calling-convention basis: [RISC-V ABIs Specification](https://riscv-non-isa.github.io/riscv-elf-psabi-doc/), RISC-V International, CC BY 4.0.*

A typical non-leaf prologue and epilogue are:

```text
example:
    addi sp, sp, -16     # allocate one aligned frame
    sw   ra, 12(sp)      # preserve caller's return address
    sw   s0, 8(sp)       # preserve callee-saved register before changing it
    addi s0, sp, 16      # optional frame pointer refers to entry sp

    # body may use 0(sp) and 4(sp) for locals or spills
    # body may call another function

    lw   s0, 8(sp)       # restore in reverse ownership sense
    lw   ra, 12(sp)
    addi sp, sp, 16      # release exactly the allocated frame
    ret
```

A frame may contain:

- saved return address and callee-saved registers;
- local variables whose address is needed;
- register spill slots when live values exceed available registers;
- outgoing stack arguments for called functions;
- alignment padding.

The **frame pointer** is optional. If used, it resides in `s0` (`x8`) and remains callee-saved. A stable frame pointer can simplify debugging and access when `sp` moves during a function, but optimized code often omits it and addresses the frame directly from `sp`.

Data below the current stack pointer is not reserved by the standard ABI. Unlike some other ABIs, code must not assume a protected red zone beneath `sp`.

#### **Arguments and Return Values** {#arguments-and-return-values}

The integer calling convention uses `a0-a7` (`x10-x17`) for the first eight integer argument slots. `a0` and `a1` are also used for return values. When argument registers are exhausted, additional values are placed on the stack according to ABI layout rules.

For RV32, a scalar no wider than XLEN usually occupies one argument register. A 64-bit scalar needs two XLEN-sized pieces and may require an aligned register pair or stack placement according to its type and available registers. Aggregates can be flattened into register-sized fields, passed in register pairs, or passed indirectly through an address. The exact rule depends on type size and the ABI, not only on the ISA.

Consider the C-like function

```c
int affine(int x, int scale, int bias) {
    return x * scale + bias;
}
```

With the `M` extension available, its argument/result mapping can be:

```text
# a0 = x, a1 = scale, a2 = bias
affine:
    mul a0, a0, a1
    add a0, a0, a2
    ret
# result is returned in a0
```

The function is a leaf and needs no stack frame: it uses only caller-saved argument registers, returns in `a0`, and does not make another call. A frame is a consequence of storage and preservation needs, not a mandatory ceremony for every procedure.

Pointers are passed as integer-register values whose width follows the ABI data model. The pointer identifies memory; the callee still needs load/store instructions to access the object. Passing an address is not the same as copying the addressed data.

Variadic functions, floating-point ABIs, vectors, and large aggregates add rules beyond this base example. When interoperating with compiled code, use the selected psABI document rather than infer behavior from register nicknames alone.

#### **Saved and Temporary Registers** {#saved-and-temporary-registers}

Calls divide register-preservation responsibility between caller and callee. "Saved" does not mean the register is automatically stored by hardware; it specifies who must preserve a value when necessary.

![The RISC-V ABI groups fixed, argument, temporary, and saved registers and assigns caller/callee preservation responsibility.](assets/riscv-register-convention.svg){fig-align="center" width="100%"}

| Register group | ABI names | Preserved across calls? | Responsibility |
|---|---|---|---|
| zero | `x0/zero` | immutable | no saved value exists |
| return address | `x1/ra` | no | caller saves if needed after another call |
| stack pointer | `x2/sp` | yes | callee restores before return |
| global/thread pointer | `x3/gp`, `x4/tp` | fixed/unallocatable | ordinary procedures do not repurpose |
| temporaries | `t0-t6` | no | caller saves live values |
| arguments/results | `a0-a7` | no | caller saves live values |
| saved registers | `s0-s11` | yes | callee saves/restores any it modifies |

A **caller-saved** register can be freely overwritten by the callee. If the caller needs its old value after the call, the caller must copy or spill it first. A **callee-saved** register lets a caller assume the value survives; any callee that changes it must save the incoming value and restore it before return.

This division reduces unnecessary memory traffic. Short-lived values around one call fit naturally in caller-saved registers; long-lived values used across many calls fit naturally in callee-saved registers. Compilers choose based on liveness and call structure.

<details>
<summary>Python contract checker: detect a callee that fails to restore saved registers</summary>

~~~python
CALLER_SAVED = {"ra", *(f"a{i}" for i in range(8)), *(f"t{i}" for i in range(7))}
CALLEE_SAVED = {"sp", *(f"s{i}" for i in range(12))}


def checked_call(registers: dict[str, int], callee) -> dict[str, int]:
    before = registers.copy()
    after = callee(registers.copy())

    changed_saved = {
        name for name in CALLEE_SAVED
        if after.get(name) != before.get(name)
    }
    if changed_saved:
        raise AssertionError(f"callee failed to restore: {sorted(changed_saved)}")
    return after


initial = {"sp": 0x8000, "s0": 99, "a0": 4, "t0": 7, "ra": 0x1004}


def valid_leaf(regs: dict[str, int]) -> dict[str, int]:
    regs["a0"] *= 2          # Return value may overwrite a0.
    regs["t0"] = 123         # Temporary may be clobbered.
    return regs


result = checked_call(initial, valid_leaf)
assert result["a0"] == 8 and result["s0"] == 99 and result["sp"] == 0x8000


def broken_callee(regs: dict[str, int]) -> dict[str, int]:
    regs["s0"] = 0           # Illegal unless restored before return.
    return regs


try:
    checked_call(initial, broken_callee)
    raise AssertionError("the ABI violation should have been detected")
except AssertionError as error:
    assert "s0" in str(error)
~~~

</details>

The checker models only preservation, not the full ABI. Real correctness also requires stack alignment, argument layout, return-value rules, and valid memory behavior.

### **RISC and CISC Design Philosophies** {#risc-and-cisc-design-philosophies}

**RISC** and **CISC** describe clusters of ISA design choices, not a binary measure of processor quality. RISC-V illustrates a RISC tendency through regular register fields, a load-store base, and mostly fixed 32-bit instructions. x86 illustrates CISC tendencies through variable-length encodings, richer addressing forms, and instructions that can combine memory access with computation.

![RISC and CISC emphasize different encoding and operation tradeoffs, while modern ISAs and processors mix techniques from both traditions.](assets/risc-cisc-tradeoffs.svg){fig-align="center" width="100%"}

| Dimension | RISC tendency | CISC tendency |
|---|---|---|
| encoding | regular fields, often fixed base length | variable length and denser special cases |
| memory operands | explicit load/store boundary | some computation directly names memory |
| operation granularity | simpler explicit steps | richer operations may combine steps |
| decode | easier boundaries and field extraction | more work to find length and operands |
| code density | may need more instructions | often fewer encoded bytes for common work |
| implementation | regular front end is convenient | translation to internal micro-operations is common |

The historical slogan "one RISC instruction per cycle" is not a modern definition. A load may miss in cache for hundreds of cycles, while an out-of-order processor may complete several instructions per cycle. Likewise, a complex architectural instruction may decode into several regular internal micro-operations and execute efficiently.

The boundary has also blurred:

- RISC-V has a compressed extension for code density and optional vector/cryptographic operations with substantial semantics;
- modern x86 cores translate instructions into internal micro-operations and use deeply pipelined, speculative execution;
- both families can use caches, branch prediction, out-of-order scheduling, SIMD/vector units, and microcode;
- compiler quality and workload behavior often matter more than the label.

An ISA design balances encoding space, code size, compiler convenience, implementation cost, compatibility, and future extension. A regular ISA can simplify a small core and verification, but it does not guarantee that every high-performance implementation is simple. A dense ISA can reduce instruction-fetch bandwidth, but it may demand more front-end decoding work.

The fairest comparison therefore separates **architectural expression** from **microarchitectural realization**. Performance must be measured on actual implementations and workloads, not inferred from the acronym.

### **Reading Compiler-Generated Assembly** {#reading-compiler-generated-assembly}

Compiler-generated assembly is easiest to read by reconstructing data flow and control flow, not by translating each mnemonic into English in isolation. Start with the ABI contract, identify loop-carried values, and then map branches and memory operations back to source-level structure.

Consider:

```c
int sum_positive(const int *a, int n) {
    int total = 0;
    for (int i = 0; i < n; ++i) {
        if (a[i] > 0) total += a[i];
    }
    return total;
}
```

One plausible optimized RV32I sequence is:

<details>
<summary>Illustrative optimized RV32I assembly for sum_positive</summary>

~~~text
# a0 = pointer a, a1 = element count n
sum_positive:
    addi a2, x0, 0          # total = 0
    bge  x0, a1, .Ldone     # if n <= 0, skip the loop

.Lloop:
    lw   a3, 0(a0)          # load current a[i]
    addi a0, a0, 4          # advance pointer by sizeof(int)
    bge  x0, a3, .Lskip     # skip addition when value <= 0
    add  a2, a2, a3         # total += value

.Lskip:
    addi a1, a1, -1         # decrement remaining count
    bne  a1, x0, .Lloop

.Ldone:
    addi a0, a2, 0          # return total in a0 (mv pseudoinstruction)
    jalr x0, 0(ra)          # return (ret pseudoinstruction)
~~~

</details>

Read it in layers:

1. **Interface:** `a0` begins as the array pointer and `a1` as `n`; `a0` must hold the return value.
2. **Live state:** `a2` is the running total, `a0` becomes a moving pointer, and `a1` becomes a remaining-element counter.
3. **Memory:** `lw a3, 0(a0)` reads one 4-byte element; incrementing the pointer by 4 implements array traversal.
4. **Control:** `.Lloop`, `.Lskip`, and `.Ldone` define the loop and conditional paths.
5. **Signed meaning:** `bge x0, a3, .Lskip` skips when zero is greater than or equal to the signed element, equivalent to `a[i] <= 0`.
6. **Procedure shape:** it is a leaf, uses only caller-saved registers, and therefore needs no stack frame.

The compiler transformed source variables. There is no explicit register for `i`; the moving pointer and decreasing count encode the same loop progress more cheaply. This is normal optimization, not loss of semantics.

Compilation options change the result:

- at `-O0`, values are often spilled to the stack and source structure remains obvious;
- at `-O2`, constants propagate, dead code disappears, loops use registers, and functions may inline;
- ISA extensions can replace instruction sequences with multiply, bit-manipulation, compressed, floating-point, or vector operations;
- debug, sanitizer, position-independent-code, and security options add instructions and metadata.

When investigating real output, record the target ISA string, ABI, compiler version, optimization level, code model, and whether the view is pre-link assembly or final disassembly. Without that context, two correct outputs can look surprisingly different.

<details>
<summary>Python helper: classify the instructions in the example by architectural role</summary>

~~~python
assembly = """
addi a2, x0, 0
bge x0, a1, .Ldone
lw a3, 0(a0)
addi a0, a0, 4
bge x0, a3, .Lskip
add a2, a2, a3
addi a1, a1, -1
bne a1, x0, .Lloop
addi a0, a2, 0
jalr x0, 0(ra)
"""

categories = {
    "add": "ALU", "addi": "ALU",
    "lw": "memory",
    "bge": "conditional control", "bne": "conditional control",
    "jalr": "indirect control",
}

counts: dict[str, int] = {}
for line in assembly.strip().splitlines():
    mnemonic = line.split()[0]
    category = categories[mnemonic]
    counts[category] = counts.get(category, 0) + 1

assert counts == {
    "ALU": 5,
    "conditional control": 3,
    "memory": 1,
    "indirect control": 1,
}
~~~

</details>

**Chapter summary.** An ISA is the software-visible contract for encoded instructions and architectural state, while assembly, ABI, and microarchitecture occupy distinct layers around it. RV32I exposes 32 integer registers, a program counter, and byte-addressed memory. Its regular 32-bit formats keep register fields fixed while adapting immediate bits for arithmetic, stores, branches, and jumps. Arithmetic operates on registers and wraps to XLEN; loads and stores alone access memory through base-plus-offset addresses; branches, `JAL`, and `JALR` select the next PC and support procedure control flow. Assembly adds labels, directives, and pseudoinstructions, then the assembler, linker, and loader turn symbolic source into executable bytes. The calling convention assigns argument, return, temporary, saved, stack, and link responsibilities so separately compiled procedures cooperate. RISC and CISC describe design tradeoffs rather than speed rankings. Reading compiler output finally reconnects all these rules by following ABI interfaces, register data flow, memory accesses, and control-flow edges. Chapter 06 will map these ISA-level state transitions onto concrete datapath components and control signals.